# 🔢 RNN, GRU et LSTM sur MNIST (PyTorch)

## Présentation du projet

Dans ce notebook, nous allons classifier des chiffres manuscrits (MNIST) en utilisant des **réseaux de neurones récurrents** — habituellement utilisés pour des séquences (texte, audio, séries temporelles).

### 💡 L'astuce : traiter une image comme une séquence
Une image MNIST fait 28×28 pixels. On va la traiter comme une **séquence de 28 lignes**, où chaque ligne est un vecteur de 28 pixels :
```
Image 28×28  →  Séquence de 28 pas de temps, chacun avec 28 features
```
Le réseau "lit" l'image ligne par ligne, du haut vers le bas, comme s'il lisait du texte !

### 🎯 Ce que vous allez apprendre :
- Implémenter **RNN, GRU et LSTM** avec PyTorch (`nn.RNN`, `nn.GRU`, `nn.LSTM`)
- Créer une classe `Dataset` personnalisée pour charger un CSV
- Construire une boucle d'entraînement complète
- Évaluer et comparer les 3 architectures

---
> ⚠️ **GPU recommandé** : `Runtime > Change runtime type > Hardware accelerator > GPU (T4)`

---
## ⚙️ PARTIE 1 — Configuration de l'environnement

On installe les bibliothèques, télécharge les données, et configure les hyperparamètres.

In [ ]:
# ── Installation des bibliothèques ────────────────────────────────────────────
!pip install -q torch torchvision pandas matplotlib seaborn scikit-learn

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os
import time
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style='darkgrid')
%matplotlib inline

print("✅ Bibliothèques importées !")

In [ ]:
# ── Téléchargement du dataset MNIST CSV ───────────────────────────────────────
url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D4/MNIST%20in%20CSV.zip"
zip_path = "mnist_csv.zip"

print("📥 Téléchargement du dataset MNIST (format CSV)...")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('./mnist_data')

print("✅ Dataset téléchargé et décompressé !")
print("Fichiers disponibles :", os.listdir('./mnist_data'))

In [ ]:
# ── Définition du Device (GPU ou CPU) ────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device utilisé : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

# ── Hyperparamètres ────────────────────────────────────────────────────────────
# Une image MNIST 28×28 est traitée comme une SÉQUENCE de 28 lignes
# Chaque ligne contient 28 pixels (= 28 features par pas de temps)

INPUT_SIZE   = 28     # Nombre de pixels par ligne (= features par timestep)
SEQUENCE_LEN = 28     # Nombre de lignes dans l'image (= nombre de timesteps)
HIDDEN_SIZE  = 128    # Nombre de neurones dans la couche récurrente
NUM_LAYERS   = 2       # Nombre de couches RNN/GRU/LSTM empilées
NUM_CLASSES  = 10     # Chiffres de 0 à 9

BATCH_SIZE    = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS    = 10

print("\n⚙️  Hyperparamètres définis :")
print(f"   INPUT_SIZE    : {INPUT_SIZE}  (pixels par ligne)")
print(f"   SEQUENCE_LEN  : {SEQUENCE_LEN}  (nombre de lignes = timesteps)")
print(f"   HIDDEN_SIZE   : {HIDDEN_SIZE}")
print(f"   NUM_LAYERS    : {NUM_LAYERS}")
print(f"   NUM_CLASSES   : {NUM_CLASSES}")
print(f"   BATCH_SIZE    : {BATCH_SIZE}")
print(f"   LEARNING_RATE : {LEARNING_RATE}")
print(f"   NUM_EPOCHS    : {NUM_EPOCHS}")

---
## 🧠 PARTIE 2 — Conception des modèles

On construit **3 architectures différentes** qui partagent la même structure générale :
```
Image (28×28) → Couche récurrente (RNN/GRU/LSTM) → Dernier état caché → Dense → 10 classes
```

### Différences entre RNN, GRU et LSTM

| Modèle | Mécanisme | Avantage | Inconvénient |
|--------|-----------|----------|---------------|
| **RNN** | État caché simple | Rapide, peu de paramètres | Oublie vite (vanishing gradient) |
| **GRU** | Portes de mise à jour/oubli | Bon compromis | Moins de mémoire que LSTM |
| **LSTM** | Cellule mémoire + 3 portes | Meilleure mémoire long-terme | Plus de paramètres, plus lent |

In [ ]:
# ── Modèle 1 : SimpleRNN ──────────────────────────────────────────────────────
class SimpleRNN(nn.Module):
    """
    RNN simple pour classifier les chiffres MNIST.
    
    Architecture :
        Input (batch, 28, 28) → RNN(hidden=128, layers=2) → dernier état → Linear(128→10)
    """
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # batch_first=True → format (batch, seq, features) plus intuitif
        self.rnn = nn.RNN(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            nonlinearity = 'tanh'  # Fonction d'activation standard du RNN
        )

        # Couche fully connected pour mapper vers les 10 classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        x : (batch_size, seq_len, input_size) = (batch, 28, 28)
        """
        batch_size = x.size(0)

        # État caché initial à zéro
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        # out : (batch, seq_len, hidden_size) — sortie à CHAQUE pas de temps
        out, _ = self.rnn(x, h0)

        # On ne garde QUE le dernier pas de temps (résumé de toute la séquence)
        out = out[:, -1, :]   # (batch, hidden_size)

        return self.fc(out)   # (batch, num_classes)


print("✅ SimpleRNN défini.")

In [ ]:
# ── Modèle 2 : SimpleGRU ──────────────────────────────────────────────────────
class SimpleGRU(nn.Module):
    """
    GRU (Gated Recurrent Unit) pour classifier les chiffres MNIST.
    
    Le GRU utilise des PORTES (gates) qui contrôlent quelle information
    garder ou oublier — cela résout le problème du vanishing gradient
    du RNN simple, tout en étant plus léger que le LSTM.
    """
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(SimpleGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.gru = nn.GRU(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        out, _ = self.gru(x, h0)
        out = out[:, -1, :]

        return self.fc(out)


print("✅ SimpleGRU défini.")

In [ ]:
# ── Modèle 3 : SimpleLSTM ─────────────────────────────────────────────────────
class SimpleLSTM(nn.Module):
    """
    LSTM (Long Short-Term Memory) pour classifier les chiffres MNIST.
    
    Le LSTM possède une CELLULE MÉMOIRE séparée (c0) en plus de
    l'état caché (h0), et 3 portes (entrée, oubli, sortie) qui
    permettent une gestion encore plus fine de l'information
    sur de longues séquences.
    """
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        # Le LSTM a DEUX états initiaux : h0 (caché) et c0 (cellule mémoire)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = out[:, -1, :]

        return self.fc(out)


print("✅ SimpleLSTM défini.")

In [ ]:
# ── Test rapide des 3 architectures ───────────────────────────────────────────
dummy_input = torch.zeros(4, SEQUENCE_LEN, INPUT_SIZE)  # batch de 4 images factices

models_test = {
    'SimpleRNN':  SimpleRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES),
    'SimpleGRU':  SimpleGRU(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES),
    'SimpleLSTM': SimpleLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES),
}

print("📐 Vérification des architectures :\n")
for name, model in models_test.items():
    output = model(dummy_input)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"   {name:<12} → sortie: {output.shape} | paramètres: {n_params:,}")

---
## 📦 PARTIE 3 — Création d'une classe Dataset personnalisée

Le fichier CSV MNIST contient une ligne par image :
```
label, pixel0, pixel1, ..., pixel783
```
Chaque image (784 pixels) doit être :
1. Extraite et normalisée (valeurs entre 0 et 1)
2. Reformée en **séquence** de (28, 28) — 28 lignes de 28 pixels

In [ ]:
# ── Classe Dataset personnalisée ──────────────────────────────────────────────
class MnistDataset(Dataset):
    """
    Dataset PyTorch pour charger MNIST depuis un fichier CSV.

    Format attendu du CSV :
        Colonne 0       : label (0-9)
        Colonnes 1-784  : valeurs des pixels (0-255)
    """
    def __init__(self, csv_path, seq_len=28, input_size=28):
        """
        Args:
            csv_path   : chemin vers le fichier CSV
            seq_len    : nombre de lignes (timesteps) par image
            input_size : nombre de pixels par ligne (features)
        """
        print(f"📂 Chargement de {csv_path}...")
        self.data = pd.read_csv(csv_path)
        self.seq_len    = seq_len
        self.input_size = input_size

        # Séparer labels et pixels
        self.labels = self.data.iloc[:, 0].values            # Colonne 'label'
        self.pixels = self.data.iloc[:, 1:].values.astype(np.float32)  # Colonnes pixels

        # Normalisation : 0-255 → 0-1 (standard pour les réseaux de neurones)
        self.pixels = self.pixels / 255.0

        print(f"✅ {len(self.labels)} images chargées.")

    def __len__(self):
        """Retourne le nombre total d'exemples dans le dataset."""
        return len(self.labels)

    def __getitem__(self, idx):
        """
        Retourne un échantillon (image, label).
        L'image est reformée de (784,) vers (28, 28) — une séquence de 28 lignes.
        """
        # Récupérer le vecteur de 784 pixels et le reformer en (28, 28)
        image = self.pixels[idx].reshape(self.seq_len, self.input_size)
        label = self.labels[idx]

        # Conversion en tenseurs PyTorch
        image_tensor = torch.tensor(image, dtype=torch.float32)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return image_tensor, label_tensor


print("✅ Classe MnistDataset définie.")

In [ ]:
# ── Trouver les fichiers CSV téléchargés ──────────────────────────────────────
csv_files = [f for f in os.listdir('./mnist_data') if f.endswith('.csv')]
print(f"📋 Fichiers CSV trouvés : {csv_files}")

# Identifier les fichiers train et test (les noms peuvent varier légèrement)
train_csv = next(f for f in csv_files if 'train' in f.lower())
test_csv  = next(f for f in csv_files if 'test' in f.lower())

train_path = os.path.join('./mnist_data', train_csv)
test_path  = os.path.join('./mnist_data', test_csv)

print(f"   Train : {train_path}")
print(f"   Test  : {test_path}")

In [ ]:
# ── Création des datasets et DataLoaders ─────────────────────────────────────
train_dataset = MnistDataset(train_path, seq_len=SEQUENCE_LEN, input_size=INPUT_SIZE)
test_dataset  = MnistDataset(test_path,  seq_len=SEQUENCE_LEN, input_size=INPUT_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n✅ DataLoaders créés :")
print(f"   Train : {len(train_dataset)} images, {len(train_loader)} batches")
print(f"   Test  : {len(test_dataset)} images, {len(test_loader)} batches")

# Vérification : afficher la forme d'un batch
sample_X, sample_y = next(iter(train_loader))
print(f"\n📐 Forme d'un batch : X={sample_X.shape}, y={sample_y.shape}")
print(f"   X : (batch_size, seq_len, input_size) = (batch, 28 lignes, 28 pixels/ligne)")

In [ ]:
# ── Visualisation de quelques exemples ────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    img, label = train_dataset[i]
    ax.imshow(img.numpy(), cmap='gray')
    ax.set_title(f'Label: {label.item()}', fontsize=10)
    ax.axis('off')
plt.suptitle('Exemples du dataset MNIST (vus comme séquences 28×28)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Chaque image est une 'séquence' de 28 lignes, chacune contenant 28 valeurs de pixels.")

---
## 🚀 PARTIE 4 — Entraînement des modèles

On définit la fonction de perte (`CrossEntropyLoss`) et l'optimiseur (`Adam`), puis on écrit une boucle d'entraînement réutilisable pour les 3 modèles.

In [ ]:
# ── Fonction d'entraînement générique ────────────────────────────────────────
def train_model(model, train_loader, num_epochs, lr, device, model_name=''):
    """
    Entraîne un modèle (RNN/GRU/LSTM) et retourne l'historique des losses.

    Args:
        model        : instance du modèle (SimpleRNN, SimpleGRU, ou SimpleLSTM)
        train_loader : DataLoader d'entraînement
        num_epochs   : nombre d'epochs
        lr           : learning rate
        device       : 'cuda' ou 'cpu'
        model_name   : nom pour l'affichage
    """
    model = model.to(device)

    # CrossEntropyLoss : standard pour la classification multi-classes
    # (combine LogSoftmax + NLLLoss en une seule fonction, numériquement stable)
    criterion = nn.CrossEntropyLoss()

    # Adam : optimiseur adaptatif, très efficace pour les RNN
    optimizer = Adam(model.parameters(), lr=lr)

    loss_history = []
    print(f"\n🏋️  Entraînement de {model_name}...")

    for epoch in range(num_epochs):
        model.train()  # Mode entraînement
        epoch_loss = 0.0

        for batch_idx, (data, targets) in enumerate(train_loader):
            data    = data.to(device)
            targets = targets.to(device)

            # 1. Réinitialiser les gradients
            optimizer.zero_grad()

            # 2. Forward pass
            outputs = model(data)

            # 3. Calcul de la loss
            loss = criterion(outputs, targets)

            # 4. Backward pass (rétropropagation)
            loss.backward()

            # 5. Mise à jour des poids
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        loss_history.append(avg_loss)

        # Afficher la loss à la fin de chaque epoch
        print(f"  [{model_name}] Epoch [{epoch+1}/{num_epochs}] — Loss: {avg_loss:.4f}")

    return model, loss_history


print("✅ Fonction d'entraînement définie.")

In [ ]:
# ── Entraînement des 3 modèles ────────────────────────────────────────────────

# 1. SimpleRNN
rnn_model = SimpleRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES)
rnn_model, rnn_losses = train_model(
    rnn_model, train_loader, NUM_EPOCHS, LEARNING_RATE, DEVICE, model_name='SimpleRNN'
)

# 2. SimpleGRU
gru_model = SimpleGRU(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES)
gru_model, gru_losses = train_model(
    gru_model, train_loader, NUM_EPOCHS, LEARNING_RATE, DEVICE, model_name='SimpleGRU'
)

# 3. SimpleLSTM
lstm_model = SimpleLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES)
lstm_model, lstm_losses = train_model(
    lstm_model, train_loader, NUM_EPOCHS, LEARNING_RATE, DEVICE, model_name='SimpleLSTM'
)

print("\n✅ Les 3 modèles sont entraînés !")

In [ ]:
# ── Comparaison des courbes de Loss ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

epochs_range = range(1, NUM_EPOCHS + 1)
ax.plot(epochs_range, rnn_losses,  'o-', color='tomato',    linewidth=2, label='SimpleRNN')
ax.plot(epochs_range, gru_losses,  's-', color='royalblue', linewidth=2, label='SimpleGRU')
ax.plot(epochs_range, lstm_losses, '^-', color='seagreen',  linewidth=2, label='SimpleLSTM')

ax.set_title("Évolution de la Loss pendant l'entraînement", fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('CrossEntropy Loss')
ax.legend()
plt.tight_layout()
plt.show()

print("💡 Une loss qui descend de façon stable indique un bon apprentissage.")

---
## 📏 PARTIE 5 — Évaluation des modèles

On calcule l'**accuracy** sur le jeu d'entraînement ET le jeu de test pour chaque modèle, et on les compare.

> 💡 Comparer accuracy train vs test permet de détecter un éventuel **surapprentissage** (overfitting).

In [ ]:
# ── Fonction de calcul d'accuracy ────────────────────────────────────────────
def check_accuracy(loader, model, device, dataset_name=''):
    """
    Calcule l'accuracy du modèle sur le dataset fourni.

    Args:
        loader       : DataLoader (train ou test)
        model        : modèle entraîné
        device       : 'cuda' ou 'cpu'
        dataset_name : nom pour l'affichage ('Train' ou 'Test')

    Returns:
        accuracy (float), all_preds (list), all_targets (list)
    """
    model.eval()  # Mode évaluation (désactive dropout, etc.)
    num_correct = 0
    num_samples = 0
    all_preds, all_targets = [], []

    with torch.no_grad():  # Pas de calcul de gradient → plus rapide
        for data, targets in loader:
            data    = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            # argmax(1) → indice de la classe avec le score le plus élevé
            _, predictions = outputs.max(1)

            num_correct += (predictions == targets).sum().item()
            num_samples += predictions.size(0)

            all_preds.extend(predictions.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    accuracy = 100 * num_correct / num_samples
    print(f"  [{dataset_name}] Accuracy : {num_correct}/{num_samples} = {accuracy:.2f}%")

    return accuracy, all_preds, all_targets


print("✅ Fonction check_accuracy définie.")

In [ ]:
# ── Évaluation des 3 modèles sur Train et Test ───────────────────────────────

results = {}

for name, model in [('SimpleRNN', rnn_model), ('SimpleGRU', gru_model), ('SimpleLSTM', lstm_model)]:
    print(f"\n📊 Évaluation de {name} :")
    train_acc, _, _                    = check_accuracy(train_loader, model, DEVICE, 'Train')
    test_acc, test_preds, test_targets = check_accuracy(test_loader,  model, DEVICE, 'Test')
    results[name] = {
        'train_acc': train_acc,
        'test_acc':  test_acc,
        'preds':     test_preds,
        'targets':   test_targets
    }

print("\n✅ Évaluation terminée pour les 3 modèles !")

In [ ]:
# ── Comparaison visuelle des accuracies ───────────────────────────────────────
names      = list(results.keys())
train_accs = [results[n]['train_acc'] for n in names]
test_accs  = [results[n]['test_acc']  for n in names]

x     = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, test_accs,  width, label='Test Accuracy',  color='tomato')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title('Comparaison des 3 architectures — Train vs Test', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrices de confusion pour les 3 modèles ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, name in zip(axes, names):
    cm = confusion_matrix(results[name]['targets'], results[name]['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=range(10), yticklabels=range(10), cbar=False)
    ax.set_title(f'{name}\n(Test acc: {results[name]["test_acc"]:.1f}%)', fontsize=11)
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.suptitle('Matrices de confusion — Comparaison RNN / GRU / LSTM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Rapport de classification détaillé (meilleur modèle) ────────────────────
best_model_name = max(results, key=lambda n: results[n]['test_acc'])
print(f"🏆 Meilleur modèle : {best_model_name} (Test accuracy: {results[best_model_name]['test_acc']:.2f}%)\n")

print(classification_report(
    results[best_model_name]['targets'],
    results[best_model_name]['preds'],
    target_names=[str(i) for i in range(10)],
    digits=3
))

In [ ]:
# ── Visualisation de quelques prédictions du meilleur modèle ────────────────
best_model = {'SimpleRNN': rnn_model, 'SimpleGRU': gru_model, 'SimpleLSTM': lstm_model}[best_model_name]
best_model.eval()

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for i, ax in enumerate(axes.flatten()):
    img, true_label = test_dataset[i]
    with torch.no_grad():
        output = best_model(img.unsqueeze(0).to(DEVICE))
        pred_label = output.argmax(1).item()

    color = 'green' if pred_label == true_label.item() else 'red'
    ax.imshow(img.numpy(), cmap='gray')
    ax.set_title(f'Réel: {true_label.item()} | Prédit: {pred_label}', fontsize=9, color=color)
    ax.axis('off')

plt.suptitle(f'Prédictions du meilleur modèle ({best_model_name})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🎓 Conclusion

Vous avez construit, entraîné et comparé **3 architectures récurrentes** sur une tâche de classification d'images :

| Partie | Ce qu'on a fait |
|--------|----------------|
| **1 - Setup** | Installé les bibliothèques, défini les hyperparamètres, configuré le device |
| **2 - Modèles** | Implémenté `SimpleRNN`, `SimpleGRU`, `SimpleLSTM` avec `nn.RNN`/`nn.GRU`/`nn.LSTM` |
| **3 - Dataset** | Créé `MnistDataset` qui transforme chaque image 784px en séquence (28, 28) |
| **4 - Entraînement** | Implémenté la boucle complète (zero_grad → forward → loss → backward → step) |
| **5 - Évaluation** | Calculé l'accuracy train/test, matrices de confusion, comparaison des 3 modèles |

### 💡 Points clés à retenir
- Traiter une image comme une **séquence de lignes** permet d'utiliser des RNN pour la vision
- Le **LSTM** et le **GRU** surpassent généralement le **RNN simple** grâce à leurs mécanismes de portes
- `batch_first=True` simplifie la manipulation des tenseurs en PyTorch
- Comparer accuracy train vs test permet de détecter l'overfitting

### 🚀 Pour aller plus loin :
- Essayer un RNN **bidirectionnel** (`bidirectional=True`)
- Tester différentes valeurs de `HIDDEN_SIZE` et `NUM_LAYERS`
- Ajouter du **Dropout** entre les couches récurrentes pour réduire l'overfitting